# Genex Week 1 CDC Brain

This notebook builds a **CDC-driven developmental interview + LLM activity generator**.

## What it does
- loads your CDC milestone table from `milestone-cdc-table.xlsx`
- assesses **all 4 domains** for every child
- starts the interview from the child's chronological age band
- moves up or down the CDC age bands to estimate a developmental range
- uses an LLM to generate **5 activities per domain**
- creates:
  - a summary table
  - an activity table
  - a JSON file for advisor review

## Important note
The CDC file only goes up to **60 months**.  
For children older than 60 months, the notebook uses the **60-month band as the starting anchor**, then interprets the result as a developmental range relative to the CDC early-childhood milestone set.

## Run order
1. Run the setup cell.
2. Run the imports + helper functions.
3. Run the CDC load cell.
4. Run the demo cases cell.
5. Run the pipeline.
6. Open the advisor feedback notebook and use the exported JSON.

In [3]:
import os
print(bool(os.environ.get("OPENAI_API_KEY")))

True


In [4]:
# If needed, run this once in the notebook environment.
%pip install pandas openpyxl openai python-dotenv ipython

Note: you may need to restart the kernel to use updated packages.


In [5]:
# ------------------------------------------------------------
# Imports + environment
# ------------------------------------------------------------
import json
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

try:
    from openai import OpenAI
except Exception:
    OpenAI = None

load_dotenv()

# ------------------------------------------------------------
# Constants
# ------------------------------------------------------------
DOMAIN_LABELS = {
    "speech_language": "Speech / Language",
    "motor": "Motor",
    "social_communication": "Social / Communication",
    "adaptive_cognitive": "Adaptive / Cognitive",
}

CATEGORY_TO_DOMAIN = {
    "social and emotial": "social_communication",
    "language and communication": "speech_language",
    "cognitive": "adaptive_cognitive",
    "movement and physical": "motor",
}

ANSWER_SCORES = {
    "yes": 1.0,
    "sometimes": 0.6,
    "with_help": 0.4,
    "not_yet": 0.0,
    "no": 0.0,
    "not_sure": 0.25,
}

# Emergency fallback only if the API call fails.
# Your normal path should be the LLM-generated activities.
FALLBACK_ACTIVITY_LIBRARY = {
    "speech_language": [
        ("Picture-supported requesting", "Use two preferred items and prompt the child to point, sign, or say what they want."),
        ("Expand their communication", "Repeat the child's word and add one more word during play or snacks."),
        ("WH-question picture practice", "Use picture cards to practice who, what, or where questions."),
        ("Routine-based language", "Practice short phrases during snack, bath, or dressing routines."),
        ("Choice making with pause time", "Offer a choice and wait 5 seconds before helping."),
    ],
    "motor": [
        ("Supported obstacle path", "Create a short safe path for stepping, reaching, and balance."),
        ("Sit-to-stand repetitions", "Practice rising from a chair or cushion during play."),
        ("Reach and place game", "Place toys at different heights to practice reaching and controlled movement."),
        ("Ball roll and stop", "Roll a ball back and forth and add stop/go cues."),
        ("Core and balance play", "Use floor play positions that encourage trunk control and balance."),
    ],
    "social_communication": [
        ("Turn-taking with favorite toy", "Use a favorite toy and practice clear my-turn/your-turn routines."),
        ("Shared attention show-and-share", "Encourage the child to show an object and look back to the caregiver."),
        ("Pretend play routine", "Model a simple pretend routine like feeding a doll or stuffed animal."),
        ("Name-response game", "Use playful name-calling and immediate reinforcement when the child responds."),
        ("Simple peer-style game", "Play a short, rule-based game with waiting and imitation."),
    ],
    "adaptive_cognitive": [
        ("Two-step helper task", "Practice a short daily routine with two simple steps."),
        ("Visual cleanup routine", "Use pictures or gestures for cleanup or dressing steps."),
        ("Matching and sorting", "Sort familiar objects by category, color, or function."),
        ("Mini task board", "Use a tiny 2-3 step board for snack, dressing, or play setup."),
        ("Cause-and-effect problem solving", "Use simple toys or household tasks that reward trial and error."),
    ],
}

# ------------------------------------------------------------
# File loading
# ------------------------------------------------------------
def find_cdc_file(filename="milestone-cdc-table.xlsx"):
    """Search a few likely locations so the notebook works from repo root or notebooks/."""
    search_roots = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
    for root in search_roots:
        candidate = root / filename
        if candidate.exists():
            return candidate.resolve()
    for root in search_roots:
        if root.exists():
            matches = list(root.rglob(filename))
            if matches:
                return matches[0].resolve()
    raise FileNotFoundError(
        f"Could not find {filename}. Put the file in the repo root or update the notebook path."
    )

def load_cdc_table(path=None):
    path = Path(path) if path else find_cdc_file()
    cdc_df = pd.read_excel(path)
    cdc_df.columns = [str(c).strip().lower() for c in cdc_df.columns]
    cdc_df = cdc_df.rename(columns={"category ": "category", "milestone ": "milestone"})
    cdc_df["category"] = cdc_df["category"].astype(str).str.strip().str.lower()
    cdc_df["milestone"] = cdc_df["milestone"].astype(str).str.strip()
    cdc_df["domain"] = cdc_df["category"].map(CATEGORY_TO_DOMAIN)
    cdc_df = cdc_df.dropna(subset=["domain"]).copy()
    cdc_df["months"] = cdc_df["months"].astype(int)
    cdc_df["question_id"] = [
        f"{row.domain}_{int(row.months)}_{i}" for i, row in enumerate(cdc_df.itertuples(), start=1)
    ]
    return cdc_df, path

def get_age_bands(cdc_df):
    return sorted(int(x) for x in cdc_df["months"].dropna().unique())

def nearest_cdc_band(cdc_df, months):
    bands = get_age_bands(cdc_df)
    capped = min(int(months), max(bands))
    eligible = [m for m in bands if m <= capped]
    return eligible[-1] if eligible else bands[0]

def get_band_questions(cdc_df, domain, month, max_q=2):
    qdf = cdc_df[(cdc_df["domain"] == domain) & (cdc_df["months"] == int(month))].copy()
    return qdf.head(max_q).to_dict("records")

# ------------------------------------------------------------
# Simulated parent answers for demo cases
# ------------------------------------------------------------
def simulated_parent_answer(true_dev_month, asked_month):
    """
    Demo-only answer simulator.
    - yes if milestone month <= underlying developmental month
    - sometimes if just above
    - with_help if clearly above but not extremely above
    - no if far above
    """
    if asked_month <= true_dev_month:
        return "yes"
    delta = asked_month - true_dev_month
    if delta <= 6:
        return "sometimes"
    if delta <= 12:
        return "with_help"
    return "no"

# ------------------------------------------------------------
# CDC interview engine
# ------------------------------------------------------------
def run_domain_interview(cdc_df, domain, chronological_months, true_dev_month, max_steps=5, max_q_per_step=2):
    """
    Start at the child's chronological age band (capped at 60 because CDC table stops there),
    ask a small number of CDC milestone questions, then move up or down until we have
    a plausible developmental range.
    """
    bands = get_age_bands(cdc_df)
    current_idx = bands.index(nearest_cdc_band(cdc_df, chronological_months))
    asked = []
    visited = []
    highest_pass_idx = None
    last_action = None

    for _ in range(max_steps):
        if current_idx < 0 or current_idx >= len(bands):
            break
        band = bands[current_idx]
        if band in visited:
            break
        visited.append(band)

        questions = get_band_questions(cdc_df, domain, band, max_q=max_q_per_step)
        scores = []

        for q in questions:
            answer = simulated_parent_answer(true_dev_month, band)
            score = ANSWER_SCORES[answer]
            scores.append(score)
            asked.append({
                "band_month": band,
                "question_id": q["question_id"],
                "question": q["milestone"],
                "answer": answer,
                "score": score,
            })

        avg_score = sum(scores) / len(scores) if scores else 0

        if avg_score >= 0.8:
            highest_pass_idx = current_idx
            last_action = "pass"
            if current_idx == len(bands) - 1:
                break
            current_idx += 1
        elif avg_score < 0.35:
            last_action = "fail"
            if current_idx == 0:
                break
            current_idx -= 1
        else:
            # If the current band is mixed / partial, drop one band lower to bound the range.
            if highest_pass_idx is None and current_idx > 0 and last_action != "partial_down":
                last_action = "partial_down"
                current_idx -= 1
            else:
                break

    band_scores = {}
    for item in asked:
        band_scores.setdefault(item["band_month"], []).append(item["score"])
    band_scores = {month: round(sum(vals) / len(vals), 2) for month, vals in band_scores.items()}

    passed = [m for m, score in sorted(band_scores.items()) if score >= 0.8]
    partial = [m for m, score in sorted(band_scores.items()) if 0.35 <= score < 0.8]
    failed = [m for m, score in sorted(band_scores.items()) if score < 0.35]

    if passed:
        lower = max(passed)
        above = [m for m in partial + failed if m > lower]
        upper = min(above) if above else bands[min(bands.index(lower) + 1, len(bands) - 1)]
    elif partial:
        upper = min(partial)
        lower = bands[max(bands.index(upper) - 1, 0)]
    else:
        upper = min(failed) if failed else bands[0]
        lower = bands[max(bands.index(upper) - 1, 0)]

    return {
        "estimated_range_months": f"{lower}-{upper}",
        "estimated_anchor_month": int(lower),
        "confidence": round(min(0.95, 0.58 + 0.06 * len(asked)), 2),
        "asked": asked,
        "band_scores": band_scores,
    }

# ------------------------------------------------------------
# LLM activity generation
# ------------------------------------------------------------
def build_child_profile(case, domain_results):
    lines = []
    for domain_key, result in domain_results.items():
        evidence = "; ".join(
            [f"{x['band_month']}m: {x['answer']} to '{x['question']}'" for x in result["asked"][:4]]
        )
        lines.append(
            f"{DOMAIN_LABELS[domain_key]} estimated around {result['estimated_range_months']} months. Evidence: {evidence}"
        )
    return "\n".join(lines)

def fallback_activities_for_domain(domain, case, result):
    items = []
    for idx, (title, instructions) in enumerate(FALLBACK_ACTIVITY_LIBRARY[domain], start=1):
        items.append({
            "activity_number": idx,
            "title": title,
            "why_this_fits": f"Matches a child with chronological age {case['chronological_age_months']} months and estimated {DOMAIN_LABELS[domain]} around {result['estimated_range_months']} months.",
            "instructions": instructions,
            "materials": "common home items",
            "duration_min": 5 + (idx % 3) * 2,
            "adaptation": "Reduce language demands, simplify the routine, and add modeling or support as needed.",
        })
    return items

def generate_activities_with_llm(case, domain_results, model="gpt-4o-mini", use_llm=True):
    """
    One OpenAI call per child.
    Returns 5 activities per domain.
    Falls back to a static set only if the API call fails.
    """
    if (not use_llm) or (OpenAI is None) or (not os.environ.get("OPENAI_API_KEY")):
        return {
            domain: fallback_activities_for_domain(domain, case, result)
            for domain, result in domain_results.items()
        }

    client = OpenAI()

    payload = {
        "child_name": case["name"],
        "chronological_age_months": case["chronological_age_months"],
        "condition": case.get("condition", ""),
        "caregiver_concern": case.get("concern_text", ""),
        "daily_time_min": case.get("daily_time_min", 10),
        "estimated_development": {
            domain: {
                "estimated_range_months": result["estimated_range_months"],
                "confidence": result["confidence"],
                "evidence": [
                    {
                        "band_month": item["band_month"],
                        "question": item["question"],
                        "answer": item["answer"],
                    }
                    for item in result["asked"][:4]
                ],
            }
            for domain, result in domain_results.items()
        },
    }

    system_prompt = '''
You are helping generate home activities for Genex.
Return exactly 5 practical, age-respectful home activities for each domain:
speech_language, motor, social_communication, adaptive_cognitive.

Rules:
- Be supportive and non-diagnostic.
- Activities must fit the child's chronological age, developmental estimate, and condition.
- Keep activities realistic for home and short enough for caregivers.
- Do not promise therapy outcomes.
- Avoid babyish wording for older children with developmental delay.
- Include adaptations for limited speech, motor challenges, or low attention.
- Return strict JSON only.

Required JSON format:
{
  "speech_language": [{"activity_number":1,"title":"...","why_this_fits":"...","instructions":"...","materials":"...","duration_min":5,"adaptation":"..."}],
  "motor": [...],
  "social_communication": [...],
  "adaptive_cognitive": [...]
}
'''

    try:
        response = client.chat.completions.create(
            model=model,
            temperature=0.3,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": json.dumps(payload)},
            ],
        )
        content = response.choices[0].message.content
        parsed = json.loads(content)

        for domain in DOMAIN_LABELS:
            if domain not in parsed or not isinstance(parsed[domain], list) or len(parsed[domain]) == 0:
                parsed[domain] = fallback_activities_for_domain(domain, case, domain_results[domain])

        return parsed

    except Exception as e:
        print(f"LLM activity generation failed for {case['case_id']}: {e}")
        return {
            domain: fallback_activities_for_domain(domain, case, result)
            for domain, result in domain_results.items()
        }

# ------------------------------------------------------------
# End-to-end case pipeline
# ------------------------------------------------------------
def run_case_pipeline(case, cdc_df, use_llm=True):
    chrono = int(case["chronological_age_months"])
    cdc_start_month = nearest_cdc_band(cdc_df, chrono)

    domain_results = {}
    for domain, true_dev_month in case["true_dev_months"].items():
        domain_results[domain] = run_domain_interview(
            cdc_df=cdc_df,
            domain=domain,
            chronological_months=chrono,
            true_dev_month=true_dev_month,
            max_steps=5,
            max_q_per_step=2,
        )

    activities_by_domain = generate_activities_with_llm(case, domain_results, use_llm=use_llm)

    return {
        "case_id": case["case_id"],
        "name": case["name"],
        "condition": case["condition"],
        "chronological_age_months": chrono,
        "cdc_start_month": cdc_start_month,
        "concern_text": case["concern_text"],
        "daily_time_min": case["daily_time_min"],
        "developmental_profile": domain_results,
        "activities_by_domain": activities_by_domain,
        "profile_text_for_prompt": build_child_profile(case, domain_results),
        "note": "CDC table only goes to 60 months, so children older than 60 months start from the 60-month band as an anchor.",
    }

def build_summary_table(results):
    rows = []
    for result in results:
        rows.append({
            "case_id": result["case_id"],
            "name": result["name"],
            "condition": result["condition"],
            "chronological_age_months": result["chronological_age_months"],
            "cdc_start_month": result["cdc_start_month"],
            "speech_range": result["developmental_profile"]["speech_language"]["estimated_range_months"],
            "motor_range": result["developmental_profile"]["motor"]["estimated_range_months"],
            "social_range": result["developmental_profile"]["social_communication"]["estimated_range_months"],
            "adaptive_range": result["developmental_profile"]["adaptive_cognitive"]["estimated_range_months"],
        })
    return pd.DataFrame(rows)

def build_activity_table(results):
    rows = []
    for result in results:
        for domain, activities in result["activities_by_domain"].items():
            for item in activities:
                rows.append({
                    "case_id": result["case_id"],
                    "name": result["name"],
                    "condition": result["condition"],
                    "domain": domain,
                    "activity_number": item.get("activity_number"),
                    "title": item.get("title"),
                    "duration_min": item.get("duration_min"),
                    "why_this_fits": item.get("why_this_fits"),
                    "materials": item.get("materials"),
                    "instructions": item.get("instructions"),
                    "adaptation": item.get("adaptation"),
                })
    return pd.DataFrame(rows)

def show_case_packet(result):
    print("=" * 100)
    print(f"{result['case_id']} | {result['name']} | condition: {result['condition']} | chronological age: {result['chronological_age_months']} months")
    print(f"CDC start month: {result['cdc_start_month']}")
    print(f"Concern: {result['concern_text']}")
    print("-" * 100)
    for domain, profile in result["developmental_profile"].items():
        print(f"{DOMAIN_LABELS[domain]} -> estimated {profile['estimated_range_months']} months | confidence {profile['confidence']}")
        for asked in profile["asked"]:
            print(f"  - [{asked['band_month']}m] {asked['question']} -> {asked['answer']}")
        print("  Activities:")
        for act in result["activities_by_domain"][domain]:
            print(f"    {act.get('activity_number')}. {act.get('title')} ({act.get('duration_min')} min)")
            print(f"       Why: {act.get('why_this_fits')}")
        print()

In [6]:
# Check that the API key is visible to the notebook.
print("OPENAI_API_KEY visible:", bool(os.environ.get("OPENAI_API_KEY")))
if not os.environ.get("OPENAI_API_KEY"):
    print("Warning: OPENAI_API_KEY is not set. The notebook will use fallback activities instead of LLM-generated activities.")

OPENAI_API_KEY visible: True


In [7]:
# Load the CDC milestone table from your repo.
cdc_df, cdc_path = load_cdc_table()
print("Loaded CDC file from:", cdc_path)
print("Age bands:", get_age_bands(cdc_df))
print("Rows:", len(cdc_df))
cdc_df.head()

Loaded CDC file from: /Users/sara/Projects/Genex/milestone-cdc-table.xlsx
Age bands: [2, 4, 6, 9, 12, 15, 18, 24, 30, 36, 48, 60]
Rows: 159


,months,category,milestone,domain,question_id
0,2,social and emotial,calms down when spoken to or picked up,social_communication,social_communication_2_1
1,2,social and emotial,looks at your face,social_communication,social_communication_2_2
2,2,social and emotial,seems happy to see you when you walk up to her,social_communication,social_communication_2_3
3,2,social and emotial,smiles when you talk to or smile at her,social_communication,social_communication_2_4
4,2,language and communication,makes sounds other than crying,speech_language,speech_language_2_5


In [8]:
# Quick sanity check on domains and milestone counts.
cdc_df.groupby(["domain", "months"]).size().reset_index(name="milestone_count").head(20)

,domain,months,milestone_count
0,adaptive_cognitive,2,2
1,adaptive_cognitive,4,2
2,adaptive_cognitive,6,3
3,adaptive_cognitive,9,2
4,adaptive_cognitive,12,2
5,adaptive_cognitive,15,2
6,adaptive_cognitive,18,2
7,adaptive_cognitive,24,3
8,adaptive_cognitive,30,4
9,adaptive_cognitive,36,2


In [9]:
# ------------------------------------------------------------
# Five demo children
# ------------------------------------------------------------
# These use simulated parent answers so you can test the full workflow fast.
# The 'true_dev_months' field is only for the demo harness.
CASES = [
    {
        "case_id": "C01",
        "name": "Emma",
        "condition": "Down syndrome",
        "chronological_age_months": 36,
        "daily_time_min": 10,
        "concern_text": "Parents want a home plan for communication, motor skills, and daily routines.",
        "true_dev_months": {
            "speech_language": 24,
            "motor": 30,
            "social_communication": 30,
            "adaptive_cognitive": 24,
        },
    },
    {
        "case_id": "C02",
        "name": "Noah",
        "condition": "Autism",
        "chronological_age_months": 60,
        "daily_time_min": 15,
        "concern_text": "Parents report limited back-and-forth play, speech delay, and rigid routines.",
        "true_dev_months": {
            "speech_language": 36,
            "motor": 60,
            "social_communication": 30,
            "adaptive_cognitive": 42,
        },
    },
    {
        "case_id": "C03",
        "name": "Liam",
        "condition": "ADHD",
        "chronological_age_months": 54,
        "daily_time_min": 10,
        "concern_text": "Parents want better daily routines, attention, task completion, and social flexibility.",
        "true_dev_months": {
            "speech_language": 36,
            "motor": 36,
            "social_communication": 24,
            "adaptive_cognitive": 36,
        },
    },
    {
        "case_id": "C04",
        "name": "Maya",
        "condition": "Speech delay",
        "chronological_age_months": 48,
        "daily_time_min": 10,
        "concern_text": "Parents are most concerned about communication frustration and limited expressive language.",
        "true_dev_months": {
            "speech_language": 30,
            "motor": 48,
            "social_communication": 42,
            "adaptive_cognitive": 48,
        },
    },
    {
        "case_id": "C05",
        "name": "Sofia",
        "condition": "Cerebral palsy",
        "chronological_age_months": 24,
        "daily_time_min": 5,
        "concern_text": "Parents want a realistic home plan for movement, communication, and independence.",
        "true_dev_months": {
            "speech_language": 12,
            "motor": 30,
            "social_communication": 12,
            "adaptive_cognitive": 9,
        },
    },
]
pd.DataFrame(CASES)[["case_id", "name", "condition", "chronological_age_months", "daily_time_min", "concern_text"]]

,case_id,name,condition,chronological_age_months,daily_time_min,concern_text
0,C01,Emma,Down syndrome,36,10,"Parents want a home plan for communication, mo..."
1,C02,Noah,Autism,60,15,"Parents report limited back-and-forth play, sp..."
2,C03,Liam,ADHD,54,10,"Parents want better daily routines, attention,..."
3,C04,Maya,Speech delay,48,10,Parents are most concerned about communication...
4,C05,Sofia,Cerebral palsy,24,5,Parents want a realistic home plan for movemen...


In [10]:
# Set to False if you want to skip OpenAI calls and use fallback activities.
USE_LLM = True

results = [run_case_pipeline(case, cdc_df, use_llm=USE_LLM) for case in CASES]
len(results)

5

In [11]:
# Summary table: one row per child.
summary_df = build_summary_table(results)
summary_df

,case_id,name,condition,chronological_age_months,cdc_start_month,speech_range,motor_range,social_range,adaptive_range
0,C01,Emma,Down syndrome,36,36,24-30,30-36,30-36,24-30
1,C02,Noah,Autism,60,60,36-48,60-60,30-36,36-48
2,C03,Liam,ADHD,54,48,36-48,36-48,24-30,36-48
3,C04,Maya,Speech delay,48,48,30-36,48-60,36-48,48-60
4,C05,Sofia,Cerebral palsy,24,24,15-18,30-36,15-18,12-15


In [12]:
# Activity table: one row per activity.
activity_df = build_activity_table(results)
activity_df.head(20)

,case_id,name,condition,domain,activity_number,title,duration_min,why_this_fits,materials,instructions,adaptation
0,C01,Emma,Down syndrome,speech_language,1,Picture Exchange Conversation,5,Encourages back-and-forth communication using ...,"Picture cards (can be homemade or printed), a ...",Use picture cards to represent common objects ...,"For limited speech, use a communication device..."
1,C01,Emma,Down syndrome,speech_language,2,Word and Action Game,5,Promotes vocabulary development through actions.,"None required, just space to move.","Say a word and perform an action (e.g., 'jump'...",Use gestures or visual prompts if Emma struggl...
2,C01,Emma,Down syndrome,speech_language,3,Story Time with Questions,5,Enhances comprehension and question-asking ski...,A children's book.,Read a short story and pause to ask Emma quest...,Use simple yes/no questions or point to pictur...
3,C01,Emma,Down syndrome,speech_language,4,Sing and Sign,5,Supports language development through music an...,None required.,Choose a simple song and incorporate hand sign...,Use visual aids or props to represent the word...
4,C01,Emma,Down syndrome,speech_language,5,Labeling Household Items,5,Encourages vocabulary building in a familiar c...,Sticky notes and a pen.,Walk around the house and label items together...,Use picture labels if Emma has difficulty with...
5,C01,Emma,Down syndrome,motor,1,Bead Stringing,5,Enhances fine motor skills through a fun activ...,"Large beads, string.",Provide large beads and a string. Encourage Em...,Use a thicker string or pipe cleaner for easie...
6,C01,Emma,Down syndrome,motor,2,Dress-Up Challenge,5,Promotes independence in dressing skills.,Loose clothing items.,Lay out loose clothing items and encourage Emm...,Assist with tricky parts like buttons or zippers.
7,C01,Emma,Down syndrome,motor,3,Twist and Turn,5,Develops hand strength and coordination.,"Containers with different lids (e.g., jars, bo...",Provide various containers with lids. Encourag...,Use larger containers for easier manipulation.
8,C01,Emma,Down syndrome,motor,4,Obstacle Course,5,Encourages gross motor skills through movement.,"Pillows, chairs, toys.","Set up a simple obstacle course with pillows, ...",Simplify the course by reducing the number of ...
9,C01,Emma,Down syndrome,motor,5,Ball Toss,5,Improves hand-eye coordination and gross motor...,Soft ball.,Use a soft ball to toss back and forth. Encour...,Use a larger ball for easier catching.


In [13]:
# Show one detailed packet.
show_case_packet(results[0])

C01 | Emma | condition: Down syndrome | chronological age: 36 months
CDC start month: 36
Concern: Parents want a home plan for communication, motor skills, and daily routines.
----------------------------------------------------------------------------------------------------
Speech / Language -> estimated 24-30 months | confidence 0.82
  - [36m] talks with you in conversation using at least two back and forth exchangess -> with_help
  - [36m] ask who or what or where or why questions like where is mommy or where is daddy -> with_help
  - [30m] says about 50 words -> sometimes
  - [30m] say two or more words toghether with one action word like doggie run -> sometimes
  Activities:
    1. Picture Exchange Conversation (5 min)
       Why: Encourages back-and-forth communication using visual supports.
    2. Word and Action Game (5 min)
       Why: Promotes vocabulary development through actions.
    3. Story Time with Questions (5 min)
       Why: Enhances comprehension and question-aski

In [14]:
# Export outputs for the advisor-feedback notebook.
out_json = Path("week1_cdc_outputs.json")
out_csv_summary = Path("week1_cdc_summary.csv")
out_csv_activities = Path("week1_cdc_activities.csv")

out_json.write_text(json.dumps(results, indent=2), encoding="utf-8")
summary_df.to_csv(out_csv_summary, index=False)
activity_df.to_csv(out_csv_activities, index=False)

print("Saved:")
print(out_json.resolve())
print(out_csv_summary.resolve())
print(out_csv_activities.resolve())

Saved:
/Users/sara/Projects/Genex/notebooks/week1_cdc_outputs.json
/Users/sara/Projects/Genex/notebooks/week1_cdc_summary.csv
/Users/sara/Projects/Genex/notebooks/week1_cdc_activities.csv
